# 04 · LLM complaint triage (Gemini Flash-Lite)

Run the cells **one at a time, top to bottom** (Shift + Enter). Wait for each `[*]` to become a number.

1. Setup → 2. Answer form → 3. Prompt v1 → 4. Prompt v2 (with the category guide) → 5. AI-calling functions → 6. Compare v1 vs v2 on 30 **dev** complaints (from training data) → 7. Run the winning prompt on all 1,000 **test** complaints (auto-saves, resumable) → 8. Progress check

In [1]:
# 1. Setup
import os, time, enum
import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types
from pydantic import BaseModel

load_dotenv()
MODEL = "gemini-flash-lite-latest"   # smaller, faster model: fewer timeouts, more generous free limits
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"),
                      http_options=types.HttpOptions(timeout=180000))   # wait up to 180 s per call

test = pd.read_csv("data/clean/test_1000.csv")
ISSUES = sorted(test["issue"].unique())
os.makedirs("outputs", exist_ok=True)

# quick connection check
print(client.models.generate_content(model=MODEL, contents="Reply with exactly: connected").text)

connected


In [2]:
# 2. The answer form the AI must fill in (fixed drop-down values, so no invented labels)
Issue = enum.Enum("Issue", {f"I{i}": v for i, v in enumerate(ISSUES)}, type=str)

class Outcome(str, enum.Enum):
    PRODUCTS = "Products and services"
    PRICE = "Price and value"
    UNDERSTANDING = "Consumer understanding"
    SUPPORT = "Consumer support"

class Severity(str, enum.Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"

class Triage(BaseModel):
    complaint_id: int
    issue: Issue
    consumer_duty_outcome: Outcome
    vulnerable: bool
    vulnerability_reason: str
    severity: Severity
    confidence: float
    summary: str

print("Answer form ready:", list(Triage.model_fields))

Answer form ready: ['complaint_id', 'issue', 'consumer_duty_outcome', 'vulnerable', 'vulnerability_reason', 'severity', 'confidence', 'summary']


In [3]:
# 3. Prompt v1 (the original instructions)
SYSTEM = f"""You are a complaints analyst at a UK credit card provider, triaging customer complaints under the FCA Consumer Duty.
For EACH complaint, fill in every field:

issue: choose exactly one of: {ISSUES}

consumer_duty_outcome: the ONE Consumer Duty outcome most at stake:
- Products and services: the product or feature did not work as it should or did not meet the customer's needs
- Price and value: fees, interest or charges the customer feels are unfair or poor value
- Consumer understanding: the customer was confused or misled by communications, terms or marketing
- Consumer support: poor service, delays, unresolved disputes, or barriers to getting help

vulnerable: true ONLY if the text mentions a driver of vulnerability from FCA guidance:
health (illness, disability, mental health), life events (bereavement, job loss, divorce, caring duties),
resilience (financial hardship, unable to pay bills, debt worries), capability (language barriers, low digital skills, confusion in older age).
Otherwise false.
vulnerability_reason: a few words naming the driver, or "" if not vulnerable.

severity: high = fraud, identity theft, a vulnerable customer, or significant financial harm; medium = clear financial or service detriment; low = minor inconvenience.

confidence: your honest probability (0 to 1) that your 'issue' choice is correct.

summary: a neutral one-line summary, max 20 words.

Use ONLY what the complaint says. Do not invent facts. Return one entry per complaint, keeping its complaint_id."""
print("Prompt v1:", len(SYSTEM), "characters")

Prompt v1: 1915 characters


In [4]:
# 4. Prompt v2 = v1 + the category guide (sub-issues from the TRAINING data, never the test set)
train_labels = pd.read_csv("data/clean/train.csv", usecols=["issue", "sub_issue"])
guide = train_labels.groupby("issue")["sub_issue"].apply(lambda s: s.value_counts().head(6).index.tolist())
ISSUE_GUIDE = "\n".join(f"- {issue}: e.g. {'; '.join(map(str, subs))}" for issue, subs in guide.items())

RULE = """
Category guide (each issue with its most common sub-issues):
""" + ISSUE_GUIDE + """
Pick the issue that matches the customer's ORIGINAL problem, using the guide above.
A disputed or unrecognised transaction that the company failed to resolve is still 'Problem with a purchase shown on your statement'.
Use 'Problem with a company's investigation into an existing problem' only when the guide's sub-issues fit (mainly credit report investigations)."""

SYSTEM_V2 = SYSTEM.replace(f"issue: choose exactly one of: {ISSUES}", f"issue: choose exactly one of: {ISSUES}\n{RULE}")
print("Prompt v2 is", len(SYSTEM_V2) - len(SYSTEM), "characters longer")

Prompt v2 is 2571 characters longer


In [5]:
# 5. Functions that call the AI (with automatic retries)
def triage_batch(rows, system):
    text = "\n\n".join(
        f"### complaint_id: {r.complaint_id}\n{' '.join(str(r.narrative).split()[:400])}"
        for r in rows.itertuples())
    t0 = time.time()
    resp = client.models.generate_content(
        model=MODEL, contents=text,
        config=types.GenerateContentConfig(
            system_instruction=system,
            response_mime_type="application/json",
            response_schema=list[Triage],
            temperature=0))
    if resp.parsed is None:
        raise ValueError("AI reply could not be read as the answer form")
    u = resp.usage_metadata
    meta = {"seconds": time.time() - t0, "in_tokens": u.prompt_token_count, "out_tokens": u.candidates_token_count}
    return [t.model_dump(mode="json") for t in resp.parsed], meta

def safe_triage(rows, system, tries=5):
    for attempt in range(tries):
        try:
            return triage_batch(rows, system)
        except Exception as e:
            msg = str(e)
            wait = 65 if "429" in msg else 20 * (attempt + 1)   # 429 = free limit hit: wait a full minute
            print(f"\n  {type(e).__name__}: {msg[:100]} -> retrying in {wait}s")
            time.sleep(wait)
    raise RuntimeError("Failed after retries. If you see 429 / RESOURCE_EXHAUSTED, the daily quota is used up - try again tomorrow.")

print("Functions ready. Model:", MODEL)

Functions ready. Model: gemini-flash-lite-latest


In [6]:
# 6. Compare prompt v1 vs v2 on 30 DEV complaints (from the training data)
dev = pd.read_csv("data/clean/train.csv").sample(30, random_state=7)

def run(df, system, batch=5):
    out = []
    for s in range(0, len(df), batch):
        res, _ = safe_triage(df.iloc[s:s+batch], system)
        out += res
        print(".", end="")
        time.sleep(5)
    print()
    return pd.DataFrame(out).merge(df[["complaint_id", "issue"]].rename(columns={"issue": "true_issue"}), on="complaint_id")

scores = {}
for name, prompt in [("v1", SYSTEM), ("v2", SYSTEM_V2)]:
    o = run(dev, prompt)
    scores[name] = (o["issue"] == o["true_issue"]).mean()
    o.to_csv(f"outputs/dev_results_{name}.csv", index=False)
    print(f"Prompt {name}: {scores[name]*100:.1f}% correct on {len(o)} dev complaints")

BEST = "v2" if scores["v2"] >= scores["v1"] else "v1"
BEST_PROMPT = SYSTEM_V2 if BEST == "v2" else SYSTEM
print("Winning prompt:", BEST)

......
Prompt v1: 60.0% correct on 30 dev complaints
......
Prompt v2: 60.0% correct on 30 dev complaints
Winning prompt: v2


In [7]:
# 7. Run the winning prompt on all 1,000 TEST complaints
#    Saves after every batch. If it stops (quota / errors), just run this cell again - it continues where it left off.
OUT = "outputs/llm_triage.csv"
done = set(pd.read_csv(OUT)["complaint_id"]) if os.path.exists(OUT) else set()
todo = test[~test["complaint_id"].isin(done)]
print(f"Already done: {len(done)} | to do: {len(todo)}")

BATCH = 5
for s in range(0, len(todo), BATCH):
    rows = todo.iloc[s:s+BATCH]
    res, meta = safe_triage(rows, BEST_PROMPT)
    part = pd.DataFrame(res)
    part = part[part["complaint_id"].isin(rows["complaint_id"])]          # keep only IDs we actually sent
    part["seconds_per_complaint"] = meta["seconds"] / len(rows)
    part["in_tokens_per_complaint"] = meta["in_tokens"] / len(rows)
    part["out_tokens_per_complaint"] = meta["out_tokens"] / len(rows)
    part["prompt_version"] = BEST
    part["model"] = MODEL
    part.to_csv(OUT, mode="a", header=not os.path.exists(OUT), index=False)
    print(f"{len(done) + s + len(rows)}/1000", end="  ")
    time.sleep(5)
print("\nFinished!")

Already done: 0 | to do: 1000
5/1000  10/1000  15/1000  20/1000  25/1000  30/1000  35/1000  40/1000  45/1000  50/1000  55/1000  60/1000  65/1000  70/1000  75/1000  80/1000  85/1000  90/1000  95/1000  100/1000  105/1000  110/1000  115/1000  120/1000  125/1000  130/1000  135/1000  140/1000  145/1000  150/1000  155/1000  160/1000  165/1000  170/1000  175/1000  180/1000  185/1000  
  ServerError: 504 DEADLINE_EXCEEDED. {'error': {'code': 504, 'message': 'Deadline expired before operation could c -> retrying in 20s
190/1000  195/1000  200/1000  205/1000  210/1000  215/1000  220/1000  225/1000  230/1000  235/1000  240/1000  245/1000  250/1000  255/1000  260/1000  265/1000  270/1000  275/1000  280/1000  285/1000  290/1000  295/1000  300/1000  305/1000  310/1000  315/1000  320/1000  325/1000  330/1000  335/1000  340/1000  345/1000  350/1000  355/1000  360/1000  365/1000  370/1000  375/1000  380/1000  385/1000  390/1000  395/1000  400/1000  405/1000  410/1000  415/1000  420/1000  425/1000  430/

In [8]:
# 8. Progress check (run any time)
res = pd.read_csv("outputs/llm_triage.csv").drop_duplicates("complaint_id")
chk = res.merge(test[["complaint_id", "issue"]].rename(columns={"issue": "true_issue"}), on="complaint_id")
print(f"Complaints triaged: {len(chk)} / 1000")
print(f"Issue accuracy so far: {(chk['issue'] == chk['true_issue']).mean()*100:.1f}%")
print(f"Flagged vulnerable: {chk['vulnerable'].mean()*100:.1f}%")
print(f"Avg seconds per complaint: {res['seconds_per_complaint'].mean():.2f} | avg tokens in/out: "
      f"{res['in_tokens_per_complaint'].mean():.0f}/{res['out_tokens_per_complaint'].mean():.0f}")

Complaints triaged: 1000 / 1000
Issue accuracy so far: 59.4%
Flagged vulnerable: 16.7%
Avg seconds per complaint: 1.03 | avg tokens in/out: 406/107
